
# PRAKTIKUM BIG DATA 2
## Pengumpulan dan Pra-pemrosesan Data (Data Acquisition & Preprocessing)

**Nama:** Ndaru Pradana Gatot Suseno  
**NIM:** 2411532007  
**Kelas:** A  


## K-1. Import Library dan Inisialisasi

Tahap ini menyiapkan library yang digunakan untuk membuat dan melakukan pra-pemrosesan dataset transaksi sintetis.

In [1]:

!pip install -q faker

import numpy as np
import pandas as pd
from faker import Faker
import random

print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 18.7 MB/s eta 0:00:00
NumPy : 2.1.3
Pandas: 2.2.3


## K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)

Dataset dibuat secara deterministik menggunakan `SEED = 42`. Beberapa masalah sengaja disisipkan, yaitu missing value, duplicate, variasi format harga, variasi format tanggal, dan perbedaan kapitalisasi teks.

In [5]:

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

fake = Faker("id_ID")
Faker.seed(SEED)

N = 500

kategori_produk = [
    "Elektronik", "Fashion", "Kesehatan",
    "Rumah Tangga", "Olahraga", "Buku"
]

metode_bayar = [
    "Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"
]

rows = []

for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(
        ["Pro", "Lite", "Max", "Basic", ""]
    )

    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice(
        [15000, 25000, 50000, 75000, 120000,
         250000, 500000, 1200000]
    )
    qty = random.randint(1, 5)

    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} "
    ]
    harga = random.choice(harga_variants)

    tgl = fake.date_between(
        start_date="-90d",
        end_date="today"
    )

    tgl_variants = [
        tgl.strftime("%Y-%m-%d"),
        tgl.strftime("%d/%m/%Y"),
        tgl.strftime("%d-%m-%Y")
    ]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)

    if random.random() < 0.3:
        metode = metode.lower()

    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating
    })

df = pd.DataFrame(rows)

# Menyisipkan missing value
for col, frac in [
    ("customer_name", 0.02),
    ("shipping_city", 0.03),
    ("payment_method", 0.015)
]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Menyisipkan 15 baris duplicate
dup_rows = df.sample(n=15, random_state=SEED)

df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)

print("Jumlah baris data mentah:", len(df))



Jumlah baris data mentah: 515


## K-3. Deteksi dan Penanganan Missing Value

Missing value diperiksa pada setiap kolom. Kolom yang bersifat wajib akan digunakan untuk menyaring baris, sedangkan informasi yang masih berguna dapat diberi nilai pengganti.

In [7]:

print(df.isnull().sum())


transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64



### Strategi Penanganan Missing Value

- `customer_name`: baris dibuang karena nama pelanggan merupakan informasi identitas transaksi.
- `payment_method`: baris dibuang karena metode pembayaran merupakan informasi inti transaksi.
- `shipping_city`: diisi dengan **"Tidak Diketahui"** karena transaksi masih dapat digunakan untuk analisis lain.
- `rating`: dibiarkan kosong karena rating bersifat opsional.


In [8]:

df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah penanganan missing value:", len(df))


Jumlah baris setelah penanganan missing value: 495


## K-4. Deteksi dan Penanganan Duplicate

Duplicate diperiksa sebelum standardisasi. Baris yang identik menunjukkan transaksi yang tercatat lebih dari satu kali dan perlu dihapus agar tidak menggandakan kontribusi transaksi pada analisis.

In [9]:

print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


## K-5. Koreksi Tipe Data dan Standardisasi Format

### K-5a. Standardisasi Teks

Kolom kategorikal distandardisasi agar perbedaan spasi dan kapitalisasi tidak menghasilkan kategori yang dianggap berbeda.

In [10]:

for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

print(df["category"].unique())
print(df["payment_method"].unique())


<StringArray>
['Buku', 'Elektronik', 'Fashion', 'Rumah Tangga', 'Kesehatan', 'Olahraga']
Length: 6, dtype: string
<StringArray>
['Kartu Kredit', 'E-Wallet', 'COD', 'Transfer Bank']
Length: 4, dtype: string


### K-5b. Membersihkan Harga

Kolom `price` memiliki beberapa bentuk teks, seperti angka biasa, format Rupiah dengan titik pemisah ribuan, angka desimal, dan spasi. Nilai tersebut dikonversi menjadi numerik.

In [11]:

def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)
df["price"].head()


,price
0,50000.0
1,500000.0
2,25000.0
3,250000.0
5,250000.0


### K-5c. Standardisasi Tanggal

Tanggal dapat muncul dalam format ISO maupun format hari-bulan-tahun. Setiap nilai diparsing menggunakan format eksplisit agar tanggal ISO tidak salah ditafsirkan.

In [12]:

def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")
df["transaction_date"].head()


,transaction_date
0,2026-07-13
1,2026-07-09
2,2026-08-25
3,2026-07-27
5,2026-09-15


### K-5d. Finalisasi Tipe Data

Tipe data numerik dan tanggal diperiksa kembali sebelum dataset diekspor.

In [13]:

df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)
df.dtypes


,0
transaction_id,object
customer_name,object
product_name,object
category,string[python]
price,float64
quantity,int64
payment_method,string[python]
transaction_date,object
shipping_city,string[python]
rating,float64


## K-6. Ekspor Dataset Bersih

Dataset yang telah melalui seluruh tahap pra-pemrosesan disimpan sebagai `transaksi_bersih.csv`, sesuai nama artefak yang ditentukan modul.

In [14]:

df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris



## O. Analisis Hasil

Dengan `SEED = 42`, dataset mentah berisi **515 baris**. Setelah penanganan missing value pada kolom wajib, data menjadi **495 baris**, kemudian setelah penghapusan **15 baris duplicate**, jumlah akhir menjadi **490 baris**.

Kolom `rating` sengaja tetap memiliki missing value karena rating bersifat opsional. Sementara itu, `shipping_city` diisi dengan `"Tidak Diketahui"` agar transaksi yang masih valid tetap dapat digunakan untuk analisis lain.

Proses ini menunjukkan bahwa data mentah tidak langsung siap digunakan. Pembersihan diperlukan untuk meningkatkan **Veracity**, yaitu tingkat kepercayaan terhadap data yang akan dianalisis.



## P. Studi Kasus

**1. Mengapa jumlah transaksi IT (515) berbeda dengan Finance (490)?**

Perbedaan terjadi karena 515 merupakan jumlah seluruh baris data mentah, sedangkan 490 merupakan jumlah data setelah proses preprocessing. Sebanyak 10 baris tereliminasi karena `customer_name` atau `payment_method` kosong, dan 15 baris duplicate dihapus.

**2. Apakah 490 baris lebih benar daripada 515 baris?**

Tidak secara otomatis. Angka 490 merupakan data yang lebih sesuai untuk analisis setelah aturan preprocessing diterapkan, tetapi keputusan tersebut tetap bergantung pada tujuan analisis dan aturan validitas yang digunakan. Konsep Veracity menekankan bahwa kualitas dan kepercayaan data lebih penting daripada sekadar jumlah baris.

**3. Mengapa rating dibiarkan missing?**

Rating merupakan informasi opsional sehingga mengisi nilai yang tidak diketahui dengan angka tebakan dapat mengubah distribusi rating dan menghasilkan kesimpulan yang bias. Untuk menghitung rata-rata rating, missing value sebaiknya diperlakukan secara eksplisit dan denominator harus menggunakan transaksi yang benar-benar memiliki rating.


## Q. Latihan

### Latihan 1 – Mengubah SEED menjadi 7

Pipeline dijalankan kembali menggunakan seed 7. Hasil jumlah baris dibandingkan dengan hasil seed 42.

In [15]:

def buat_dataset(seed):
    np.random.seed(seed)
    random.seed(seed)

    fake = Faker("id_ID")
    Faker.seed(seed)

    rows = []

    for i in range(1, 501):
        trx_id = f"TRX{i:05d}"
        nama = fake.name()
        produk = fake.word().capitalize() + " " + random.choice(
            ["Pro", "Lite", "Max", "Basic", ""]
        )
        kategori = random.choice(kategori_produk)
        harga_dasar = random.choice(
            [15000, 25000, 50000, 75000, 120000,
             250000, 500000, 1200000]
        )
        qty = random.randint(1, 5)

        harga = random.choice([
            str(harga_dasar),
            f"Rp{harga_dasar:,}".replace(",", "."),
            f"{harga_dasar}.0",
            f" {harga_dasar} "
        ])

        tgl = fake.date_between(start_date="-90d", end_date="today")
        tanggal = random.choice([
            tgl.strftime("%Y-%m-%d"),
            tgl.strftime("%d/%m/%Y"),
            tgl.strftime("%d-%m-%Y")
        ])

        metode = random.choice(metode_bayar)
        if random.random() < 0.3:
            metode = metode.lower()

        if random.random() < 0.2:
            kategori = kategori.upper() + " "

        rows.append({
            "transaction_id": trx_id,
            "customer_name": nama,
            "product_name": produk.strip(),
            "category": kategori,
            "price": harga,
            "quantity": qty,
            "payment_method": metode,
            "transaction_date": tanggal,
            "shipping_city": fake.city(),
            "rating": random.choice([1, 2, 3, 4, 5, None, None])
        })

    d = pd.DataFrame(rows)

    for col, frac in [
        ("customer_name", 0.02),
        ("shipping_city", 0.03),
        ("payment_method", 0.015)
    ]:
        idx = d.sample(frac=frac, random_state=seed).index
        d.loc[idx, col] = np.nan

    dup = d.sample(n=15, random_state=seed)
    d = pd.concat([d, dup], ignore_index=True)
    d = d.sample(frac=1, random_state=seed).reset_index(drop=True)

    return d

df_seed7_raw = buat_dataset(7)

raw_42 = 515
raw_7 = len(df_seed7_raw)

df_seed7_clean = df_seed7_raw.dropna(
    subset=["customer_name", "payment_method"]
).copy()
df_seed7_clean["shipping_city"] = df_seed7_clean[
    "shipping_city"
].fillna("Tidak Diketahui")
df_seed7_clean = df_seed7_clean.drop_duplicates()

clean_42 = 490
clean_7 = len(df_seed7_clean)

hasil_seed = pd.DataFrame({
    "SEED": [42, 7],
    "Baris_mentah": [raw_42, raw_7],
    "Baris_bersih": [clean_42, clean_7]
})

display(hasil_seed)


,SEED,Baris_mentah,Baris_bersih
0,42,515,490
1,7,515,490



**Analisis Latihan 1:** jumlah baris pada dataset seed 42 dan seed 7 tetap sama, yaitu **515 baris mentah** dan **490 baris bersih**. Hal ini terjadi karena ukuran dataset (`N = 500`), jumlah duplicate (15 baris), serta proporsi missing value ditentukan secara tetap. Yang berubah adalah isi dan distribusi nilai karena proses acak menggunakan seed yang berbeda.


### Latihan 2 – Menambahkan `is_valid_price`

Kolom `is_valid_price` digunakan untuk menandai harga yang valid apabila `price > 0`.

In [16]:

df["is_valid_price"] = df["price"] > 0

print("Jumlah harga valid  :", df["is_valid_price"].sum())
print("Jumlah harga invalid:", (~df["is_valid_price"]).sum())

display(
    df[["transaction_id", "price", "is_valid_price"]].head(10)
)


Jumlah harga valid  : 490
Jumlah harga invalid: 0


,transaction_id,price,is_valid_price
0,TRX00305,50000.0,True
1,TRX00500,500000.0,True
2,TRX00442,25000.0,True
3,TRX00154,250000.0,True
5,TRX00132,250000.0,True
6,TRX00205,120000.0,True
7,TRX00010,1200000.0,True
8,TRX00326,150000.0,True
9,TRX00248,1200000.0,True
10,TRX00466,50000.0,True



**Analisis Latihan 2:** seluruh nilai `price` yang berhasil dikonversi menjadi numerik kemudian diperiksa menggunakan kondisi `price > 0`. Jika jumlah harga invalid bernilai **0**, berarti tidak ditemukan harga nol atau negatif pada dataset bersih. Kolom validasi ini membantu memastikan bahwa nilai harga layak digunakan untuk analisis.


### Latihan 3 – Jumlah Transaksi per Category

Distribusi transaksi dihitung menggunakan `value_counts()` pada dataset yang sudah bersih.

In [17]:

jumlah_per_category = df["category"].value_counts()

display(
    jumlah_per_category.to_frame("jumlah_transaksi")
)


,jumlah_transaksi
category,
Olahraga,97
Kesehatan,91
Elektronik,89
Buku,82
Fashion,66
Rumah Tangga,65



**Analisis Latihan 3:** hasil `value_counts()` menunjukkan distribusi jumlah transaksi pada setiap kategori produk. Kategori dengan jumlah transaksi terbesar merupakan kategori yang paling sering muncul dalam dataset bersih, sedangkan kategori dengan jumlah terkecil memiliki kontribusi transaksi paling rendah. Hasil ini dapat digunakan sebagai gambaran awal distribusi transaksi marketplace.
